In [2]:
!pip install elasticsearch --upgrade -i https://mirrors.aliyun.com/pypi/simple/

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/


In [3]:
# 构建mapping
index_mapping = {
    "mappings":{
        "properties":{
            "docContext":{
                "type":"text"
            },
            "docEmbedding":{
                "type":"dense_vector",
                "dims":1536,
                "index":True,
                "similarity":"cosine"
            },
            "title":{
                "type":"text"
            },
            "dataTime":{
                "type":"text"
            }
        }
    
    }
}

In [4]:
from elasticsearch import Elasticsearch

# 连接远程的ES库
es_tool = Elasticsearch(
    hosts = [
        "http://localhost:9200"
    ],
    basic_auth = ("elastic", "Dhf8IOJU"),
    verify_certs=True
)


In [6]:
es_tool.indices.exists(index = "test")

HeadApiResponse(False)

In [7]:
def create_index(es, index_name, index_mapping):
    try:
        if es.indices.exists(index=index_name):
            print(f"Index {index_name} already exists !!")
            return True
        else:
            es.indices.create(index=index_name, body=index_mapping)
            print(f"Index {index_name} Create Succefull !!")
            return True
    except Exception as e:
        print(f"Index {index_name} Create Failed, error is {e}")
        return False

In [8]:
create_index(es_tool, "test", index_mapping)

Index test Create Succefull !!


True

In [9]:
es_tool.indices.exists(index="test")

HeadApiResponse(True)

In [11]:
es_tool.indices.get_mapping(index="test")

ObjectApiResponse({'test': {'mappings': {'properties': {'dataTime': {'type': 'text'}, 'docContext': {'type': 'text'}, 'docEmbedding': {'type': 'dense_vector', 'dims': 1536, 'index': True, 'similarity': 'cosine', 'index_options': {'type': 'bbq_hnsw', 'm': 16, 'ef_construction': 100, 'rescore_vector': {'oversample': 3.0}}}, 'title': {'type': 'text'}}}}})

In [12]:
es_tool.indices.delete(index="test")

ObjectApiResponse({'acknowledged': True})

In [13]:
es_tool.indices.exists(index="test")

HeadApiResponse(False)

In [16]:
import json
docs = []
with open("../data_handle/doc1.json", "r") as f:
    for line in f:
        docs.append(json.loads(line))

In [17]:
docs[1]

{'subTitle': '',
 'dataTime': '2024-12-24',
 'contentText': '山东省供销合作社第五次代表大会在济南召开林武周乃翔提出要求王宇燕讲话\u3000\u3000□记者 李子路 刘兵 报道本报济南12月23日讯 今天上午，山东省供销合作社第五次代表大会在济南召开。会前，省委书记林武，省委副书记、省长周乃翔提出要求。省委副书记王宇燕，中华全国供销合作总社党组成员、理事会副主任张梅林出席大会并讲话。林武指出，供销合作社是推动农业农村发展的一支重要力量，近年来，我省供销合作社系统紧紧围绕全省改革发展大局，主动担当作为、积极探索创新，在推动农业农村现代化中发挥了重要作用。各级党委和政府要深入学习贯彻习近平总书记关于供销合作社工作的重要指示批示和视察山东重要讲话精神，进一步加强对供销合作社工作的领导和支持，推动全省供销合作社工作高质量发展。全省供销合作社系统要认真践行新发展理念，牢记为农服务根本宗旨，主动融入全省“三农”工作大局，持续打造服务农民生产生活和促进现代农业发展的综合平台，当好党和政府密切联系农民群众的桥梁纽带，更好助力打造乡村振兴齐鲁样板，为扛牢“走在前、挑大梁”的使命担当、奋力谱写中国式现代化山东篇章作出新的更大贡献。周乃翔指出，近年来，全省供销合作社系统认真贯彻落实党中央、国务院决策部署，落实省委工作要求，在推动农业增效、促进农民增收、繁荣城乡发展等方面作出积极贡献。要深入学习贯彻党的二十届三中全会精神和习近平总书记视察山东重要讲话精神，牢记为农服务根本宗旨，持续提升农资流通服务水平，深化农业社会化服务，着力构建重要农产品应急保障体系，推动供销合作事业高质量发展，为促进城乡融合发展、推进乡村全面振兴、建设现代化强省贡献更大力量。会议要求，全省供销合作社系统要始终坚持从“三农”工作大局出发，深入学习运用“千万工程”经验，扎根农业农村，在加快建设农业强省中展现新担当。要做强为农服务主责主业，扎实做好农资保供、应急保障、商贸流通等工作，助力建设更高水平“齐鲁粮仓”。要深化综合改革，拓展基层网络，做强龙头企业，强化科技赋能，增强供销合作事业发展动能。要强化自身建设，深入推进全面从严治社、从严治企，确保供销合作社规范安全发展。各级各有关部门要积极支持供销合作社工作，切实形成推动全省供销合作事业高质量发展合力。省领导范华

In [20]:
def add_doc(es, index_name, document, data_id):
    try:
        es.index(index=index_name, id=data_id, document=document)
        print(f"Add data to Index {index_name}, Doc id is {data_id},Success!!")
        return True
    except Exception as e:
        print(f"Add data to Index {index_name} Failed, erros is {e}")
        return False

In [21]:
document = {
    "docContext":docs[1]["contentText"],
    "docEmbedding":None,
    "title":docs[1]["title"],
    "dataTime":docs[1]["dataTime"]
}
add_doc(es_tool, index_name="test", document=document, data_id="test:1")

Add data to Index test, Doc id is test:1,Success!!


True

In [22]:
def get_doc(es, index_name, data_id):
    try:
        resp = es.get(index=index_name, id=data_id)
        return resp["_source"]
    except Exception as e:
        print(f"Index {index_name}, get data {data_id} Failed!!")

In [23]:
get_doc(es_tool, index_name="test", data_id="test:1")

{'docContext': '山东省供销合作社第五次代表大会在济南召开林武周乃翔提出要求王宇燕讲话\u3000\u3000□记者 李子路 刘兵 报道本报济南12月23日讯 今天上午，山东省供销合作社第五次代表大会在济南召开。会前，省委书记林武，省委副书记、省长周乃翔提出要求。省委副书记王宇燕，中华全国供销合作总社党组成员、理事会副主任张梅林出席大会并讲话。林武指出，供销合作社是推动农业农村发展的一支重要力量，近年来，我省供销合作社系统紧紧围绕全省改革发展大局，主动担当作为、积极探索创新，在推动农业农村现代化中发挥了重要作用。各级党委和政府要深入学习贯彻习近平总书记关于供销合作社工作的重要指示批示和视察山东重要讲话精神，进一步加强对供销合作社工作的领导和支持，推动全省供销合作社工作高质量发展。全省供销合作社系统要认真践行新发展理念，牢记为农服务根本宗旨，主动融入全省“三农”工作大局，持续打造服务农民生产生活和促进现代农业发展的综合平台，当好党和政府密切联系农民群众的桥梁纽带，更好助力打造乡村振兴齐鲁样板，为扛牢“走在前、挑大梁”的使命担当、奋力谱写中国式现代化山东篇章作出新的更大贡献。周乃翔指出，近年来，全省供销合作社系统认真贯彻落实党中央、国务院决策部署，落实省委工作要求，在推动农业增效、促进农民增收、繁荣城乡发展等方面作出积极贡献。要深入学习贯彻党的二十届三中全会精神和习近平总书记视察山东重要讲话精神，牢记为农服务根本宗旨，持续提升农资流通服务水平，深化农业社会化服务，着力构建重要农产品应急保障体系，推动供销合作事业高质量发展，为促进城乡融合发展、推进乡村全面振兴、建设现代化强省贡献更大力量。会议要求，全省供销合作社系统要始终坚持从“三农”工作大局出发，深入学习运用“千万工程”经验，扎根农业农村，在加快建设农业强省中展现新担当。要做强为农服务主责主业，扎实做好农资保供、应急保障、商贸流通等工作，助力建设更高水平“齐鲁粮仓”。要深化综合改革，拓展基层网络，做强龙头企业，强化科技赋能，增强供销合作事业发展动能。要强化自身建设，深入推进全面从严治社、从严治企，确保供销合作社规范安全发展。各级各有关部门要积极支持供销合作社工作，切实形成推动全省供销合作事业高质量发展合力。省领导范华平、陈平、梅建华出席大会。省供销社党组书记、理事会主任张传忠代表省供销社第四届理事会作了题

In [24]:
def delete_doc(es, index_name, data_id):
    try:
        es.delete(index=index_name, id=data_id)
        print(f"Index {index_name}, delete id {data_id} Success!!")
        return True
    except Exception as e:
        print(f"Index {index_name}, delete id {data_id} Failed!!")
        return False

In [25]:
delete_doc(es_tool, "test", "test:1")

Index test, delete id test:1 Success!!


True

In [26]:
get_doc(es_tool, index_name="test", data_id="test:1")

Index test, get data test:1 Failed!!
